# 从有限差分到非线性求根：logit 偏置校准

这份 Notebook 用同一个标量问题连接两章数值方法。我们有 12 个固定 logit $z_i$，给它们共同加一个偏置 $b$，再观察平均 sigmoid 概率：

$$
p_i(b)=\sigma(z_i+b),\qquad
F(b)=\frac{1}{n}\sum_i p_i(b)-0.62.
$$

有限差分章节用附近的 $F(b\pm h)$ 检查 $F'(b)$；非线性方程章节继续求 $F(b)=0$。这只是确定性教学夹具，不训练模型，也不把“均值匹配”当成完整概率校准。

In [1]:
from pathlib import Path
import hashlib
import json
import os

import numpy as np
from scipy.special import expit

EXPECTED_FIXTURE_SHA256 = "f78fab203ad67476d26937fb7ae84e57b6aefaa86e4045f01b3b1c85b3b78bbc"

def locate_fixture():
    candidates = [
        Path("logit-calibration-fixture.json"),
        Path(os.environ["ML_ATLAS_CALIBRATION_FIXTURE_PATH"]) if os.environ.get("ML_ATLAS_CALIBRATION_FIXTURE_PATH") else None,
        Path("public/datasets/numerical-methods/logit-calibration-fixture.json"),
    ]
    for candidate in candidates:
        if candidate is not None and candidate.is_file():
            return candidate
    raise FileNotFoundError("Place logit-calibration-fixture.json beside this Notebook or set ML_ATLAS_CALIBRATION_FIXTURE_PATH")

fixture_path = locate_fixture()
observed_sha = hashlib.sha256(fixture_path.read_bytes()).hexdigest()
assert observed_sha == EXPECTED_FIXTURE_SHA256
fixture = json.loads(fixture_path.read_text(encoding="utf-8"))
assert fixture["contractVersion"] == "numerical-methods-batch-3-v1"

logits = np.asarray(fixture["logits"], dtype=np.float64)
target_rate = float(fixture["targetPositiveRate"])
probe_bias = float(fixture["finiteDifferenceProbeBias"])
bracket = tuple(float(value) for value in fixture["rootBracket"])
newton_start = float(fixture["newtonStart"])
secant_starts = tuple(float(value) for value in fixture["secantStarts"])

assert logits.shape == (12,)
assert np.isfinite(logits).all()
assert 0 < target_rate < 1
assert bracket[0] < bracket[1]
print(f"fixture={fixture_path.name} · logits={len(logits)} · target={target_rate:.2f} · sha256={observed_sha[:12]}…")

fixture=logit-calibration-fixture.json · logits=12 · target=0.62 · sha256=f78fab203ad6…


## 1. 先固定同一个残差与解析导数

共同偏置不会改变样本排序，只会整体移动 sigmoid 概率。残差 $F(b)$ 在本例中严格单调递增，因为

$$
F'(b)=\frac{1}{n}\sum_i p_i(b)(1-p_i(b))>0.
$$

因此只要区间两端异号，根就是唯一的。解析导数用于核对有限差分；后面 Newton 法也会使用同一导数。

In [2]:
def probabilities(bias):
    return expit(logits + float(bias))

def residual(bias):
    return float(probabilities(bias).mean() - target_rate)

def residual_derivative(bias):
    p = probabilities(bias)
    return float(np.mean(p * (1.0 - p)))

baseline_mean = float(probabilities(0.0).mean())
probe_residual = residual(probe_bias)
probe_exact_derivative = residual_derivative(probe_bias)
print(f"mean probability at b=0 = {baseline_mean:.12f}")
print(f"F({probe_bias:.2f}) = {probe_residual:.12f}")
print(f"analytic F'({probe_bias:.2f}) = {probe_exact_derivative:.12f}")
print(f"bracket residuals = [{residual(bracket[0]):.12f}, {residual(bracket[1]):.12f}]")

mean probability at b=0 = 0.501519161870
F(0.35) = -0.060786988105
analytic F'(0.35) = 0.163098254400
bracket residuals = [-0.569264153475, 0.326647222251]


## 2. 同一导数，不同步长

前向差分只看 $F(b+h)-F(b)$，主截断误差为 $O(h)$；中心差分比较两侧，主截断误差为 $O(h^2)$：

$$
D_f(h)=\frac{F(b+h)-F(b)}{h},\qquad
D_c(h)=\frac{F(b+h)-F(b-h)}{2h}.
$$

步长从 $10^{-1}$ 缩到 $10^{-12}$。误差先下降，再因为相消与浮点舍入上升，所以“最小的 $h$”不是“最好的 $h$”。

In [3]:
def forward_difference(fn, point, step):
    return (fn(point + step) - fn(point)) / step

def central_difference(fn, point, step):
    return (fn(point + step) - fn(point - step)) / (2.0 * step)

step_exponents = list(range(-1, -13, -1))
step_sweep = []
for exponent in step_exponents:
    step = 10.0 ** exponent
    forward = forward_difference(residual, probe_bias, step)
    central = central_difference(residual, probe_bias, step)
    step_sweep.append({
        "exponent": exponent,
        "step": step,
        "forward": forward,
        "forwardAbsoluteError": abs(forward - probe_exact_derivative),
        "central": central,
        "centralAbsoluteError": abs(central - probe_exact_derivative),
    })

best_forward = min(step_sweep, key=lambda row: row["forwardAbsoluteError"])
best_central = min(step_sweep, key=lambda row: row["centralAbsoluteError"])
print(" exponent      forward error      central error")
for row in step_sweep:
    print(f"{row['exponent']:>9d}  {row['forwardAbsoluteError']:.3e}       {row['centralAbsoluteError']:.3e}")
print(f"best sampled forward h = {best_forward['step']:.0e} · error = {best_forward['forwardAbsoluteError']:.3e}")
print(f"best sampled central h = {best_central['step']:.0e} · error = {best_central['centralAbsoluteError']:.3e}")

 exponent      forward error      central error
       -1  7.258e-04       4.945e-05
       -2  6.818e-05       4.947e-07
       -3  6.774e-06       4.947e-09
       -4  6.770e-07       4.873e-11
       -5  6.769e-08       2.339e-12
       -6  6.826e-09       5.317e-11
       -7  6.083e-10       5.317e-11
       -8  1.382e-08       8.274e-09
       -9  5.279e-08       5.279e-08
      -10  1.693e-07       1.693e-07
      -11  6.492e-06       6.492e-06
      -12  6.492e-06       6.200e-05
best sampled forward h = 1e-07 · error = 6.083e-10
best sampled central h = 1e-05 · error = 2.339e-12


In [4]:
finite_summary = {
    "contractVersion": "numerical-methods-batch-3-v1",
    "outputId": "finite-difference-calibration-summary",
    "fixtureSha256": EXPECTED_FIXTURE_SHA256,
    "logitCount": int(len(logits)),
    "targetPositiveRate": target_rate,
    "baselineMeanProbability": baseline_mean,
    "probeBias": probe_bias,
    "probeResidual": probe_residual,
    "analyticDerivative": probe_exact_derivative,
    "stepSweep": step_sweep,
    "bestSampledForward": best_forward,
    "bestSampledCentral": best_central,
}

output_dir = Path(os.environ.get("ML_ATLAS_NUMERICAL_BATCH3_OUTPUT_DIR", "batch-3-outputs"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "finite-difference-calibration-summary.json").write_text(
    json.dumps(finite_summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(json.dumps({
    "probeBias": finite_summary["probeBias"],
    "analyticDerivative": finite_summary["analyticDerivative"],
    "bestForwardStep": best_forward["step"],
    "bestForwardError": best_forward["forwardAbsoluteError"],
    "bestCentralStep": best_central["step"],
    "bestCentralError": best_central["centralAbsoluteError"],
}, ensure_ascii=False, indent=2))

{
  "probeBias": 0.35,
  "analyticDerivative": 0.1630982543997438,
  "bestForwardStep": 1e-07,
  "bestForwardError": 6.082833126086484e-10,
  "bestCentralStep": 1e-05,
  "bestCentralError": 2.3393509351876673e-12
}


## 3. 把同一个残差压到零

现在不再问某点斜率，而是找满足 $F(b)=0$ 的偏置。

- 二分法维护异号区间，每次把区间宽度减半。
- Newton 法使用 $b_{k+1}=b_k-F(b_k)/F'(b_k)$。
- 割线法用两个函数值近似导数，不需要解析 $F'(b)$。

下面的循环同时记录迭代点、残差、步长或区间宽度，以及函数/导数调用次数。停止条件固定为残差不超过 $10^{-12}$；二分法还会检查区间宽度，开放方法还会检查步长。

In [5]:
RESIDUAL_TOLERANCE = 1e-12
POSITION_TOLERANCE = 1e-10
MAX_ITERATIONS = 80

def bisection_trace(fn, initial_bracket):
    a, b = initial_bracket
    fa, fb = fn(a), fn(b)
    function_evaluations = 2
    if np.sign(fa) == np.sign(fb):
        raise ValueError("Bisection requires a sign-changing bracket")
    trace = []
    for iteration in range(1, MAX_ITERATIONS + 1):
        midpoint = (a + b) / 2.0
        fm = fn(midpoint)
        function_evaluations += 1
        trace.append({
            "iteration": iteration,
            "x": midpoint,
            "residual": fm,
            "bracketWidth": b - a,
        })
        if abs(fm) <= RESIDUAL_TOLERANCE or (b - a) / 2.0 <= POSITION_TOLERANCE:
            break
        if np.sign(fa) == np.sign(fm):
            a, fa = midpoint, fm
        else:
            b, fb = midpoint, fm
    return {
        "method": "bisection",
        "root": trace[-1]["x"],
        "residual": abs(trace[-1]["residual"]),
        "iterationCount": len(trace),
        "functionEvaluations": function_evaluations,
        "derivativeEvaluations": 0,
        "trace": trace,
    }

def newton_trace(fn, derivative, start):
    x = start
    function_evaluations = 0
    derivative_evaluations = 0
    trace = []
    for iteration in range(MAX_ITERATIONS + 1):
        fx = fn(x)
        function_evaluations += 1
        trace.append({"iteration": iteration, "x": x, "residual": fx})
        if abs(fx) <= RESIDUAL_TOLERANCE:
            break
        dfx = derivative(x)
        derivative_evaluations += 1
        if not np.isfinite(dfx) or abs(dfx) <= np.finfo(float).eps:
            raise ArithmeticError("Newton derivative is too small")
        step = -fx / dfx
        trace[-1]["step"] = step
        if abs(step) <= POSITION_TOLERANCE:
            break
        x = x + step
        if not np.isfinite(x) or abs(x) > 100:
            raise ArithmeticError("Newton iteration left the safeguarded teaching domain")
    return {
        "method": "newton",
        "root": trace[-1]["x"],
        "residual": abs(trace[-1]["residual"]),
        "iterationCount": len(trace) - 1,
        "functionEvaluations": function_evaluations,
        "derivativeEvaluations": derivative_evaluations,
        "trace": trace,
    }

def secant_trace(fn, starts):
    previous, current = starts
    previous_value, current_value = fn(previous), fn(current)
    function_evaluations = 2
    trace = [
        {"iteration": 0, "x": previous, "residual": previous_value},
        {"iteration": 1, "x": current, "residual": current_value},
    ]
    for iteration in range(2, MAX_ITERATIONS + 2):
        denominator = current_value - previous_value
        if abs(denominator) <= np.finfo(float).eps:
            raise ArithmeticError("Secant denominator is too small")
        step = -current_value * (current - previous) / denominator
        next_value_x = current + step
        if not np.isfinite(next_value_x) or abs(next_value_x) > 100:
            raise ArithmeticError("Secant iteration left the safeguarded teaching domain")
        next_value = fn(next_value_x)
        function_evaluations += 1
        trace[-1]["step"] = step
        trace.append({"iteration": iteration, "x": next_value_x, "residual": next_value})
        previous, previous_value = current, current_value
        current, current_value = next_value_x, next_value
        if abs(next_value) <= RESIDUAL_TOLERANCE or abs(step) <= POSITION_TOLERANCE:
            break
    return {
        "method": "secant",
        "root": trace[-1]["x"],
        "residual": abs(trace[-1]["residual"]),
        "iterationCount": len(trace) - 2,
        "functionEvaluations": function_evaluations,
        "derivativeEvaluations": 0,
        "trace": trace,
    }

bisection = bisection_trace(residual, bracket)
newton = newton_trace(residual, residual_derivative, newton_start)
secant = secant_trace(residual, secant_starts)
solver_results = [bisection, newton, secant]

for result in solver_results:
    print(
        f"{result['method']:>9s}: root={result['root']:.12f} · "
        f"|F|={result['residual']:.3e} · iterations={result['iterationCount']} · "
        f"F calls={result['functionEvaluations']} · F' calls={result['derivativeEvaluations']}"
    )

bisection: root=0.730290740321 · |F|=1.002e-12 · iterations=37 · F calls=39 · F' calls=0
   newton: root=0.730290740298 · |F|=4.649e-12 · iterations=3 · F calls=4 · F' calls=4
   secant: root=0.730290740327 · |F|=0.000e+00 · iterations=5 · F calls=7 · F' calls=0


## 4. 快不等于无条件可靠

本例的 $F'(b)$ 始终为正，Newton 从 $b=0$ 很快。但在极端负偏置处，sigmoid 饱和会让导数很小，裸 Newton 步可能直接冲出合理范围。二分法则必须先得到异号区间；目标率若超出可达到范围，区间不会异号。

实际求解器常使用 safeguarded Newton、Brent 法、阻尼或信赖域：先保留可靠边界，再在局部接受更快的开放步。这里固定记录失败条件，不把一次成功轨迹误写成全局保证。

In [6]:
failure_checks = {}
try:
    bisection_trace(residual, (-4.0, -3.0))
except ValueError as error:
    failure_checks["invalidBracket"] = str(error)

try:
    newton_trace(residual, residual_derivative, -8.0)
except ArithmeticError as error:
    failure_checks["saturatedNewton"] = str(error)

assert set(failure_checks) == {"invalidBracket", "saturatedNewton"}
print(json.dumps(failure_checks, ensure_ascii=False, indent=2))

{
  "invalidBracket": "Bisection requires a sign-changing bracket",
  "saturatedNewton": "Newton iteration left the safeguarded teaching domain"
}


In [7]:
root_summary = {
    "contractVersion": "numerical-methods-batch-3-v1",
    "outputId": "nonlinear-calibration-summary",
    "fixtureSha256": EXPECTED_FIXTURE_SHA256,
    "logitCount": int(len(logits)),
    "targetPositiveRate": target_rate,
    "root": float(newton["root"]),
    "meanProbabilityAtRoot": float(probabilities(newton["root"]).mean()),
    "derivativeAtRoot": residual_derivative(newton["root"]),
    "residualTolerance": RESIDUAL_TOLERANCE,
    "positionTolerance": POSITION_TOLERANCE,
    "bracket": list(bracket),
    "solvers": solver_results,
    "failureChecks": failure_checks,
}

(output_dir / "nonlinear-calibration-summary.json").write_text(
    json.dumps(root_summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(json.dumps({
    "root": root_summary["root"],
    "meanProbabilityAtRoot": root_summary["meanProbabilityAtRoot"],
    "derivativeAtRoot": root_summary["derivativeAtRoot"],
    "solverIterations": {row["method"]: row["iterationCount"] for row in solver_results},
    "failureChecks": failure_checks,
}, ensure_ascii=False, indent=2))

{
  "root": 0.730290740297536,
  "meanProbabilityAtRoot": 0.619999999995351,
  "derivativeAtRoot": 0.15594569798407712,
  "solverIterations": {
    "bisection": 37,
    "newton": 3,
    "secant": 5
  },
  "failureChecks": {
    "invalidBracket": "Bisection requires a sign-changing bracket",
    "saturatedNewton": "Newton iteration left the safeguarded teaching domain"
  }
}


## 5. 读完后应保留的边界

1. 有限差分是用函数值验证局部变化率，步长要在截断误差与浮点误差之间折中。
2. 求根算法比较的是可靠性、局部速度、导数需求和调用成本，不是只比迭代次数。
3. 匹配平均概率只解决这个标量方程，不代表每个概率都已校准；真实应用还需要独立数据、可靠性图和任务指标。
4. 下一章的数值优化会把“让残差为零”扩展为“沿着目标函数寻找更小值”，并加入下降、线搜索和停止判断。